In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
RRM+ Part-1 — Rank-1 Hierarchical Trainer (Lightweight & Accurate)
-------------------------------------------------------------------
Inputs  : data/fft_dataset.csv (+ optional data/Aplog_all_channels.csv)
Outputs : models_hierarchical/
          - stage1_xgb.pkl (calibrated)  + selector_stage1.pkl + scaler_stage1.pkl
          - stage2_lgbm.pkl (calibrated) + scaler_stage2.pkl
          - pca_fft.pkl
          - feat_cols.json
          - metrics + confusion matrices + chained inference sample
Author  : IIT-PID RRM+ Team
"""

import os, json, time, logging, warnings, argparse
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

warnings.filterwarnings("ignore")

# ---------------- CLI ----------------
def parse_args():
    p = argparse.ArgumentParser("Rank-1 Hierarchical Trainer")
    p.add_argument("--fft_csv", default="data/fft_dataset.csv", help="Path to fft_dataset.csv")
    p.add_argument("--ap_csv",  default="data/Aplog_all_channels.csv", help="Path to Aplog_all_channels.csv")
    p.add_argument("--save_dir", default="models_hierarchical", help="Where to save artifacts")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--pca_dims", type=int, default=10)
    p.add_argument("--test_size", type=float, default=0.2)
    p.add_argument("--cv_folds", type=int, default=5)
    p.add_argument("--use_gpu", type=int, default=int(os.getenv("USE_GPU", "0")))
    return p.parse_args()

args = parse_args()
SEED = args.seed
SAVE_DIR = Path(args.save_dir); SAVE_DIR.mkdir(exist_ok=True, parents=True)

# ---------------- Logging ----------------
log_file = SAVE_DIR / f"train_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(log_file), logging.StreamHandler()],
)
log = logging.getLogger("rank1")

# ---------------- Imports (ML) ----------------
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_curve, auc, f1_score)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import make_pipeline
from imblearn.combine import SMOTETomek

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

# ---------------- Helpers ----------------
TARGET_ALLOWED = {"wifi", "BLE", "ZigBee", "Microwave", "FHSS"}

def save_cm(cm: np.ndarray, labels, title, path):
    fig, ax = plt.subplots(figsize=(9,7))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right"); ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        s = cm[i].sum() if cm[i].sum() else 1
        for j in range(cm.shape[1]):
            pct = 100.0 * cm[i, j] / s
            ax.text(j, i, f"{cm[i,j]}\n({pct:.1f}%)",
                    ha="center", va="center",
                    color=("white" if cm[i,j] > cm.max()/2 else "black"), fontsize=9)
    ax.set_title(title); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)
    log.info(f"Saved {path}")

def band_to_num(b):
    if pd.isna(b): return 0
    s = str(b)
    if s.startswith("2.4"): return 0
    if s.startswith("5"):   return 1
    if s.startswith("6"):   return 2
    try: return int(float(s))
    except: return 0

def normalize_type(val):
    if pd.isna(val): return "wifi"
    s = str(val).strip()
    if s.lower() in ("none","nan",""): return "wifi"
    if "," in s: s = s.split(",")[0].strip()
    return s

def auc_pr_from_probs(y_true, p_pos):
    prec, rec, _ = precision_recall_curve(y_true, p_pos)
    return float(auc(rec, prec))

# ---------------- Load data ----------------
fft = pd.read_csv(args.fft_csv)
log.info(f"Loaded FFT: {len(fft):,} rows, {fft.shape[1]} cols")
ap  = None
if Path(args.ap_csv).exists():
    ap = pd.read_csv(args.ap_csv)
    log.info(f"Loaded AP : {len(ap):,} rows, {ap.shape[1]} cols")
else:
    log.warning("AP CSV not found. Proceeding with FFT-only features.")

# ---------------- FFT features ----------------
fft_bin_cols = [c for c in fft.columns if c.startswith("fft_bin_")]
if not fft_bin_cols:
    raise RuntimeError("No fft_bin_* columns found in FFT dataset.")

# Basic compact stats
bins = fft[fft_bin_cols].values.astype(np.float32)
fft_feat = pd.DataFrame({
    "timestamp": fft["timestamp"],
    "ap_id":     fft["ap_id"],
    "center_mhz": fft.get("center_mhz", pd.Series(index=fft.index, dtype=float)),
    "bw_mhz":     fft.get("bw_mhz", pd.Series(index=fft.index, dtype=float)),
    "duty":       fft.get("duty", pd.Series(index=fft.index, dtype=float)),
    "power_dbm":  fft.get("power_dbm", pd.Series(index=fft.index, dtype=float)),
    "fft_mean":   bins.mean(axis=1),
    "fft_std":    bins.std(axis=1),
    "fft_max":    bins.max(axis=1),
    "fft_min":    bins.min(axis=1),
})
# PCA latent (autoencoder-lite)
pca = PCA(n_components=args.pca_dims, random_state=SEED)
fft_latent = pca.fit_transform(bins)
for i in range(fft_latent.shape[1]):
    fft_feat[f"fft_latent_{i}"] = fft_latent[:, i]

# Label from FFT
fft_feat["fft_label"] = fft.get("label", pd.Series(index=fft.index, dtype=object))

# ---------------- Merge with AP (optional, robust to column case) ----------------
def lower_cols(df): df.columns = [c.lower() for c in df.columns]; return df
fft_l = lower_cols(fft_feat.copy())
if ap is not None:
    ap_l = lower_cols(ap.copy())
    # Map uppercase AP columns to expected names if present
    rename_map = {
        "timestamp":"timestamp", "ap_id":"ap_id", "band":"band",
        "channel":"channel", "channel_width":"channel_width_mhz",
        "channel_width_mhz":"channel_width_mhz",
        "tx_power_dbm":"tx_power_dbm", "noise_floor_dbm":"noise_floor_dbm",
        "avg_client_snr_db":"avg_client_snr_db", "throughput_avg_mbps":"throughput_avg_mbps",
        "p95_retry_pct":"p95_retry_pct", "mean_qoe":"mean_qoe",
        "mean_rssi_dbm":"mean_rssi_dbm"
    }
    ap_l = ap_l.rename(columns=rename_map)
    keep_cols = [c for c in [
        "timestamp","ap_id","band","channel","channel_width_mhz","tx_power_dbm",
        "noise_floor_dbm","avg_client_snr_db","throughput_avg_mbps","p95_retry_pct",
        "mean_qoe","mean_rssi_dbm"
    ] if c in ap_l.columns]
    ap_slim = ap_l[keep_cols].copy()
    merged = fft_l.merge(ap_slim, on=["timestamp","ap_id"], how="inner")
else:
    merged = fft_l.copy()

# Target
y_target = merged["fft_label"].apply(normalize_type) if "fft_label" in merged.columns else pd.Series("wifi", index=merged.index)
y_target = y_target.apply(lambda s: s if s in TARGET_ALLOWED else "wifi")

# Band numeric
if "band" in merged.columns:
    merged["band_num"] = merged["band"].apply(band_to_num)
else:
    merged["band_num"] = 0

# Final feature columns
feat_cols = [c for c in [
    # AP-ish
    "band_num","channel","channel_width_mhz","tx_power_dbm","noise_floor_dbm",
    "avg_client_snr_db","throughput_avg_mbps","p95_retry_pct","mean_qoe","mean_rssi_dbm",
    # FFT meta
    "center_mhz","bw_mhz","duty","power_dbm",
    # FFT compact
    "fft_mean","fft_std","fft_max","fft_min",
] if c in merged.columns] + [c for c in merged.columns if c.startswith("fft_latent_")]

log.info(f"Feature columns ({len(feat_cols)}): {feat_cols}")

X_all = merged[feat_cols].astype(np.float32).fillna(0).values
y_all = y_target.values.astype(str)

# ---------------- Stage-1: Wi-Fi vs Non-Wi-Fi ----------------
log.info("="*80); log.info("[Stage-1] Wi-Fi vs Non-Wi-Fi")
y1_bin = np.where(y_all == "wifi", "wifi", "nonwifi")
le1 = LabelEncoder()
y1_enc = le1.fit_transform(y1_bin)

X1_tr, X1_te, y1_tr, y1_te = train_test_split(
    X_all, y1_enc, test_size=args.test_size, stratify=y1_enc, random_state=SEED
)

scaler1 = RobustScaler()
X1_tr_s = scaler1.fit_transform(X1_tr).astype(np.float32)
X1_te_s = scaler1.transform(X1_te).astype(np.float32)

xgb_tree_method = "gpu_hist" if args.use_gpu else "hist"
params1 = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method=xgb_tree_method,
    n_estimators=2000,
    learning_rate=0.06,
    max_depth=7,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.6,
    gamma=0.3,
    random_state=SEED,
    n_jobs=-1,
)
model1_raw = XGBClassifier(**params1)
model1_raw.fit(
    X1_tr_s, y1_tr,
    eval_set=[(X1_tr_s, y1_tr), (X1_te_s, y1_te)],
    early_stopping_rounds=100,
    verbose=False,
)

# Feature selection by importance (median)
selector1 = SelectFromModel(model1_raw, threshold="median", prefit=True)
X1_tr_sel = selector1.transform(X1_tr_s)
X1_te_sel = selector1.transform(X1_te_s)

# Refit + Calibrate
model1_refit = XGBClassifier(**{**params1, "n_estimators": model1_raw.best_iteration or params1["n_estimators"]})
model1_refit.fit(X1_tr_sel, y1_tr, eval_set=[(X1_te_sel, y1_te)], verbose=False)
model1 = CalibratedClassifierCV(model1_refit, cv=3, method="sigmoid")
model1.fit(X1_tr_sel, y1_tr)

# Metrics
p1 = model1.predict_proba(X1_te_sel)[:,1]
auc_pr1 = auc_pr_from_probs(y1_te, p1)
y1_pred = (p1 >= 0.5).astype(int)
cm1 = confusion_matrix(y1_te, y1_pred, labels=[0,1])
save_cm(cm1, le1.inverse_transform([0,1]), "Stage-1 Confusion", SAVE_DIR/"confusion_stage1.png")
rep1 = classification_report(y1_te, y1_pred, target_names=le1.classes_, output_dict=True, digits=4)
log.info(f"[Stage-1] Acc={rep1['accuracy']:.4f} | Macro-F1={rep1['macro avg']['f1-score']:.4f} | AUC(PR)={auc_pr1:.4f}")

# CV Macro-F1
skf = StratifiedKFold(n_splits=args.cv_folds, shuffle=True, random_state=SEED)
cv_scores=[]
for i,(tr,va) in enumerate(skf.split(X_all, y1_enc),1):
    Xtr_s = scaler1.transform(X_all[tr]); Xva_s = scaler1.transform(X_all[va])
    Xtr_sel = selector1.transform(Xtr_s);  Xva_sel = selector1.transform(Xva_s)
    clf = XGBClassifier(**{**params1, "n_estimators": model1_raw.best_iteration or params1["n_estimators"]})
    clf.fit(Xtr_sel, y1_enc[tr], verbose=False)
    pred = clf.predict(Xva_sel)
    cv_scores.append(f1_score(y1_enc[va], pred, average="macro"))
log.info(f"[Stage-1] CV Macro-F1: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# ---------------- Stage-2: Non-Wi-Fi Subclasses ----------------
log.info("="*80); log.info("[Stage-2] BLE / ZigBee / Microwave / FHSS")
mask_nonwifi = (y_all != "wifi")
X2 = X_all[mask_nonwifi]
y2 = y_all[mask_nonwifi]
le2 = LabelEncoder()
y2_enc = le2.fit_transform(y2)
classes2 = list(le2.classes_)
log.info(f"Stage-2 classes: {classes2}")

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2_enc, test_size=args.test_size, stratify=y2_enc, random_state=SEED
)

scaler2 = RobustScaler()
X2_tr_s = scaler2.fit_transform(X2_tr).astype(np.float32)
X2_te_s = scaler2.transform(X2_te).astype(np.float32)

# Balance with SMOTETomek
smt = SMOTETomek(random_state=SEED)
X2_bal, y2_bal = smt.fit_resample(X2_tr_s, y2_tr)
log.info(f"[Stage-2] After SMOTETomek: {len(X2_bal):,} samples")

# LightGBM model (fast & small)
model2_raw = LGBMClassifier(
    n_estimators=1800,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.5,
    objective="multiclass",
    random_state=SEED,
    n_jobs=-1,
)
model2_raw.fit(
    X2_bal, y2_bal,
    eval_set=[(X2_te_s, y2_te)],
    eval_metric="multi_logloss",
    callbacks=[],  # could add early stopping via callbacks if needed
)

# Calibrate Stage-2
model2 = CalibratedClassifierCV(model2_raw, cv=3, method="sigmoid")
model2.fit(X2_bal, y2_bal)

# Metrics
p2 = model2.predict_proba(X2_te_s)
y2_pred = np.argmax(p2, axis=1)
cm2 = confusion_matrix(y2_te, y2_pred)
save_cm(cm2, classes2, "Stage-2 Confusion", SAVE_DIR/"confusion_stage2.png")
rep2 = classification_report(y2_te, y2_pred, target_names=classes2, output_dict=True, digits=4)
log.info(f"[Stage-2] Acc={rep2['accuracy']:.4f} | Macro-F1={rep2['macro avg']['f1-score']:.4f}")

# ---------------- Chained Evaluation ----------------
log.info("="*80); log.info("[Chained] End-to-End on new split")
X_tr_all, X_te_all, y_tr_all, y_te_all = train_test_split(
    X_all, y_all, test_size=args.test_size, stratify=y_all, random_state=SEED
)
# Stage-1
X_te_all_s = scaler1.transform(X_te_all); X_te_all_sel = selector1.transform(X_te_all_s)
p_nonwifi = model1.predict_proba(X_te_all_sel)[:,1]
bin_pred = np.where(p_nonwifi >= 0.5, "nonwifi", "wifi")

final_preds = []
idx_nw = np.where(bin_pred == "nonwifi")[0]
sub_preds = np.array([], dtype=str)
if len(idx_nw):
    X_sub = scaler2.transform(X_te_all[idx_nw])
    sub = model2.predict(X_sub)
    sub_preds = le2.inverse_transform(sub)

j=0
for i in range(len(X_te_all)):
    if bin_pred[i] == "wifi":
        final_preds.append("wifi")
    else:
        final_preds.append(sub_preds[j] if j < len(sub_preds) else classes2[0]); j+=1
final_preds = np.array(final_preds)
labels_full = ["wifi"] + classes2
cm_chain = confusion_matrix(y_te_all, final_preds, labels=labels_full)
save_cm(cm_chain, labels_full, "Chained Confusion", SAVE_DIR/"confusion_chained.png")
rep_chain = classification_report(y_te_all, final_preds, labels=labels_full, output_dict=True, digits=4)
log.info(f"[Chained] Acc={rep_chain['accuracy']:.4f} | Macro-F1={rep_chain['macro avg']['f1-score']:.4f}")

# ---------------- Persist Artifacts ----------------
joblib.dump(pca, SAVE_DIR/"pca_fft.pkl")
joblib.dump((scaler1, selector1, le1), SAVE_DIR/"stage1_preproc.pkl")
joblib.dump(model1, SAVE_DIR/"stage1_xgb.pkl")
joblib.dump((scaler2, le2), SAVE_DIR/"stage2_preproc.pkl")
joblib.dump(model2, SAVE_DIR/"stage2_lgbm.pkl")
with open(SAVE_DIR/"feat_cols.json","w") as f: json.dump(feat_cols, f, indent=2)

# ---------------- Metrics JSON ----------------
proc = psutil.Process(os.getpid())
summary = {
    "data": {
        "fft_rows": int(len(fft)),
        "merged_rows": int(len(merged)),
        "n_features": int(len(feat_cols))
    },
    "stage1": {
        "classes": list(le1.classes_),
        "accuracy": float(rep1["accuracy"]),
        "macro_f1": float(rep1["macro avg"]["f1-score"]),
        "auc_pr": float(auc_pr1),
        "cv_macro_f1_mean": float(np.mean(cv_scores)),
        "cv_macro_f1_std": float(np.std(cv_scores)),
    },
    "stage2": {
        "classes": classes2,
        "accuracy": float(rep2["accuracy"]),
        "macro_f1": float(rep2["macro avg"]["f1-score"]),
    },
    "chained": {
        "accuracy": float(rep_chain["accuracy"]),
        "macro_f1": float(rep_chain["macro avg"]["f1-score"]),
    },
    "system": {
        "cpu_percent": float(psutil.cpu_percent(interval=None)),
        "ram_mb": float(proc.memory_info().rss / 1024**2),
        "log_file": str(log_file)
    }
}
with open(SAVE_DIR/"training_summary.json","w") as f:
    json.dump(summary, f, indent=2)
log.info(json.dumps(summary, indent=2))

# ---------------- Inference Wrapper ----------------
def load_artifacts(save_dir: str):
    save_dir = Path(save_dir)
    pca_fft      = joblib.load(save_dir/"pca_fft.pkl")
    scaler1, selector1, le1 = joblib.load(save_dir/"stage1_preproc.pkl")
    model1       = joblib.load(save_dir/"stage1_xgb.pkl")
    scaler2, le2 = joblib.load(save_dir/"stage2_preproc.pkl")
    model2       = joblib.load(save_dir/"stage2_lgbm.pkl")
    feat_cols    = json.loads((save_dir/"feat_cols.json").read_text())
    return pca_fft, scaler1, selector1, le1, model1, scaler2, le2, model2, feat_cols

def chained_infer(ap_row: dict, artifacts=None, save_dir: str = None, p_thresh: float = 0.5):
    """
    ap_row: dict with keys exactly matching feat_cols (missing keys default to 0)
    returns: final_label, p_nonwifi, stage2_probs (dict or None)
    """
    if artifacts is None:
        artifacts = load_artifacts(save_dir or args.save_dir)
    pca_fft, scaler1, selector1, le1, model1, scaler2, le2, model2, feat_cols = artifacts

    x = np.array([[ap_row.get(k, 0) for k in feat_cols]], dtype=np.float32)
    # Stage-1
    x1 = scaler1.transform(x)
    x1_sel = selector1.transform(x1)
    p_nonwifi = float(model1.predict_proba(x1_sel)[0,1])
    if p_nonwifi < p_thresh:
        return "wifi", p_nonwifi, None
    # Stage-2
    x2 = scaler2.transform(x)
    p2 = model2.predict_proba(x2)[0]
    idx = int(np.argmax(p2))
    label2 = le2.inverse_transform([idx])[0]
    return label2, p_nonwifi, {cls: float(p2[i]) for i, cls in enumerate(le2.classes_)}

# Demo inference on one real sample if available
try:
    sample = dict(zip(feat_cols, X_all[0]))
    pred_label, p_nonwifi, detail = chained_infer(sample, save_dir=str(SAVE_DIR))
    with open(SAVE_DIR/"inference_example.json","w") as f:
        json.dump({"pred_label":pred_label, "p_nonwifi":p_nonwifi, "stage2_detail":detail}, f, indent=2)
    log.info(f"Saved inference example → {SAVE_DIR}/inference_example.json")
except Exception as e:
    log.warning(f"Could not run demo inference: {e}")

log.info("✅ Done. Artifacts in %s", SAVE_DIR.resolve())


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
RRM+ Part-1 Classifier Evaluation & Resource Audit
--------------------------------------------------
Evaluates trained hierarchical models on held-out FFT data.

Outputs:
  • reports/metrics_detailed.csv        → precision/recall/F1 per class
  • reports/confusion_matrix.png        → combined confusion matrix
  • reports/inference_confidence.csv    → confidence + duty + center_freq + bw
  • reports/resource_usage.json         → CPU/RAM for training vs inference

Author: IIT-PID RRM+ Team
"""

import os, time, json, psutil, logging
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import joblib

# ---------------- Paths ----------------
MODEL_DIR = Path("models_hierarchical")
REPORT_DIR = Path("reports"); REPORT_DIR.mkdir(exist_ok=True)
DATA_FFT = Path("data/fft_dataset.csv")

# ---------------- Logging ----------------
log_file = REPORT_DIR / "evaluate.log"
logging.basicConfig(level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(log_file), logging.StreamHandler()])
log = logging.getLogger("eval")

# ---------------- Load Artifacts ----------------
def load_artifacts():
    pca_fft      = joblib.load(MODEL_DIR/"pca_fft.pkl")
    scaler1, selector1, le1 = joblib.load(MODEL_DIR/"stage1_preproc.pkl")
    model1       = joblib.load(MODEL_DIR/"stage1_xgb.pkl")
    scaler2, le2 = joblib.load(MODEL_DIR/"stage2_preproc.pkl")
    model2       = joblib.load(MODEL_DIR/"stage2_lgbm.pkl")
    feat_cols    = json.loads((MODEL_DIR/"feat_cols.json").read_text())
    return pca_fft, scaler1, selector1, le1, model1, scaler2, le2, model2, feat_cols

# ---------------- Plot Confusion ----------------
def plot_cm(cm, labels, path):
    fig, ax = plt.subplots(figsize=(9,7))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        s = cm[i].sum() if cm[i].sum() else 1
        for j in range(cm.shape[1]):
            pct = 100.0 * cm[i, j] / s
            ax.text(j, i, f"{cm[i,j]}\n({pct:.1f}%)",
                    ha="center", va="center",
                    color=("white" if cm[i,j] > cm.max()/2 else "black"), fontsize=9)
    plt.title("Confusion Matrix – Combined Hierarchical Model")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()
    log.info(f"Saved confusion matrix → {path}")

# ---------------- Evaluate ----------------
def main():
    log.info("Loading artifacts & data...")
    artifacts = load_artifacts()
    pca_fft, scaler1, selector1, le1, model1, scaler2, le2, model2, feat_cols = artifacts
    fft = pd.read_csv(DATA_FFT)
    log.info(f"Loaded FFT dataset: {len(fft):,} rows")

    # Prepare X, y, and relevant columns
    fft_cols = [c for c in fft.columns if c.startswith("fft_bin_")]
    bins = fft[fft_cols].values.astype(np.float32)
    fft_latent = pca_fft.transform(bins)
    for i in range(fft_latent.shape[1]):
        fft[f"fft_latent_{i}"] = fft_latent[:, i]

    # Add fallback numeric cols
    for c in ["center_mhz","bw_mhz","duty","power_dbm"]:
        if c not in fft.columns: fft[c] = 0.0

    X = fft[[c for c in feat_cols if c in fft.columns]].astype(np.float32).fillna(0).values
    y = fft["label"].fillna("wifi").astype(str).values
    y = np.array([s if s in {"wifi","BLE","ZigBee","Microwave","FHSS"} else "wifi" for s in y])

    # Measure inference latency + resource usage
    proc = psutil.Process(os.getpid())
    mem_before = proc.memory_info().rss
    t0 = time.perf_counter()

    # Stage-1
    X1_s = scaler1.transform(X)
    X1_sel = selector1.transform(X1_s)
    p_nonwifi = model1.predict_proba(X1_sel)[:,1]
    bin_pred = np.where(p_nonwifi >= 0.5, "nonwifi", "wifi")

    # Stage-2 on predicted non-wifi subset
    idx_nw = np.where(bin_pred == "nonwifi")[0]
    preds = np.array(["wifi"]*len(X), dtype=object)
    if len(idx_nw):
        X2_s = scaler2.transform(X[idx_nw])
        sub_probs = model2.predict_proba(X2_s)
        sub_labels = le2.inverse_transform(np.argmax(sub_probs, axis=1))
        for i, label in zip(idx_nw, sub_labels):
            preds[i] = label

    inf_time = time.perf_counter() - t0
    mem_after = proc.memory_info().rss
    ram_infer_mb = (mem_after - mem_before) / 1024**2

    # Combined evaluation
    labels_full = ["wifi"] + list(le2.classes_)
    cm = confusion_matrix(y, preds, labels=labels_full)
    plot_cm(cm, labels_full, REPORT_DIR/"confusion_combined.png")
    rep = classification_report(y, preds, labels=labels_full, output_dict=True, digits=4)
    pd.DataFrame(rep).T.to_csv(REPORT_DIR/"metrics_detailed.csv")

    # Confidence export
    conf_table = pd.DataFrame({
        "pred_label": preds,
        "p_nonwifi": p_nonwifi,
        "duty_cycle": fft.get("duty", pd.Series([0]*len(fft))),
        "center_freq_mhz": fft.get("center_mhz", pd.Series([0]*len(fft))),
        "bandwidth_mhz": fft.get("bw_mhz", pd.Series([0]*len(fft))),
        "power_dbm": fft.get("power_dbm", pd.Series([0]*len(fft)))
    })
    conf_table.to_csv(REPORT_DIR/"inference_confidence.csv", index=False)

    # Resource summary (CPU, RAM, latency)
    cpu_percent = psutil.cpu_percent(interval=None)
    summary = {
        "n_samples": len(fft),
        "classes": labels_full,
        "accuracy": float(rep["accuracy"]),
        "macro_f1": float(rep["macro avg"]["f1-score"]),
        "weighted_f1": float(rep["weighted avg"]["f1-score"]),
        "cpu_percent": cpu_percent,
        "ram_inference_mb": ram_infer_mb,
        "avg_latency_ms_per_sample": (inf_time/len(fft))*1000,
        "total_inference_time_s": inf_time,
    }
    with open(REPORT_DIR/"resource_usage.json","w") as f:
        json.dump(summary, f, indent=2)
    log.info(json.dumps(summary, indent=2))
    log.info(f"✅ Reports written to {REPORT_DIR.resolve()}")

if __name__ == "__main__":
    main()
